# Shark Knowledge Medallion — Composición y Transición de Cada Capa

En este notebook exploramos en detalle la composición de cada capa de la arquitectura medallón y cómo los datos transicionan entre ellas.

---

## 1. Panorama general de la arquitectura medallón

La arquitectura medallón organiza los datos en cinco etapas progresivas:

```
                       ┌─────────────────────┐
                       │       BRONZE        │
                       │     (JSON crudo)    │
                       └────────┬────────────┘
                                │
                                ▼
                       ┌─────────────────────┐
                       │       SILVER        │
                       │     (validación)    │
                       └────────┬────────────┘
                                │
                                ▼
                       ┌─────────────────────┐
                       │      MERGE          │
                       │     (integración)   │
                       └────────┬────────────┘
                                │
                                ▼
                       ┌─────────────────────┐
                       │      WAREHOUSE      │
                       │      (DuckDB)       │
                       └────────┬────────────┘
                                │
                                ▼
                       ┌────────────────────┐
                       │       GOLD         │
                       │   (conocimiento)   │
                       └────────────────────┘
```

### Principios clave

- **Determinismo**: mismo input → mismo output en cada corrida.
- **Idempotencia**: reprocesar un lote produce filas_nuevas = 0.
- **Pivote Wikipedia**: si no hay página válida, la especie no avanza.
- **GBIF como augmentation**: taxonomía (solo EXACT) + ocurrencias agregadas.
- **Clave natural**: `species_key = slugify(scientific_name)`.

In [1]:
import os
from pathlib import Path

# Ubicar raíz del repo (donde está src/) y posicionarse ahí
while not (Path.cwd() / "src").exists():
    os.chdir("..")
    if Path.cwd().parent == Path.cwd():
        raise RuntimeError("No se encontró el directorio src/ — ejecuta el notebook desde el repo")

In [2]:
from src.pipeline import run_pipeline
from src.config.seed_species import SEED_SPECIES
from src.resolver.slugify import slugify
from src.bronze.fetcher import (
    fetch_gbif_taxonomy, fetch_gbif_occurrences,
    fetch_wikipedia_summary, fetch_wikipedia_extracts, persist_bronze,
)
from src.resolver.wikipedia import resolve_wikipedia_title, ResolutionStatus
from src.silver.models import WikipediaPage, GBIFTaxonomy, GBIFOccurrence
from src.silver.validator import (
    validate_wikipedia, validate_gbif_taxonomy, validate_gbif_occurrence, RejectionReason,
)
from src.merge.merger import merge_species
from src.warehouse.loader import Warehouse
# Cerrar conexiones DuckDB previas del mismo proceso
Warehouse.close_all()
from src.gold.models import Species, SpeciesChunk, SpeciesMetadata
from src.gold.indexer import build_species_from_merged, chunk_species, search
import json

/home/luisgm/Documents/Diplomado AI Engineer - Augemented Humans/Mis repos/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(f'Lote semilla: {len(SEED_SPECIES)} especies')
print()
print('Ejemplo de species_key (clave natural que atraviesa todas las capas):')
for nombre in ['Carcharodon carcharias', 'Rhincodon typus', 'Galeocerdo cuvier']:
    print(f'  {nombre:30s} -> {slugify(nombre)}')

Lote semilla: 18 especies

Ejemplo de species_key (clave natural que atraviesa todas las capas):
  Carcharodon carcharias         -> carcharodon_carcharias
  Rhincodon typus                -> rhincodon_typus
  Galeocerdo cuvier              -> galeocerdo_cuvier


---
## 2. Bronze Layer — Raw JSON

### Propósito

Capturar la respuesta **íntegra** de cada API en archivos JSON con timestamp, sin transformar ni filtrar. Incluso errores HTTP se persisten tal cual.

### Estructura de salida

```
data/bronze/
├── gbif_taxonomy/lote_1__2026-07-22T10-00-00Z.json
├── gbif_occurrence/lote_1__2026-07-22T10-02-00Z.json
└── wikipedia/lote_1__2026-07-22T10-01-00Z.json
```

Cada archivo usa un **envelope de metadatos**:

```json
{
  "source": "wikipedia",
  "lote": 1,
  "ingested_at": "2026-07-22T10:00:00Z",
  "species_queried": [...],
  "raw_responses": [{"query": "...", "response": {...}}]
}
```

### 2.1 Query a GBIF Species Match — taxonomía

Endpoint: `GET https://api.gbif.org/v1/species/match?name={scientificName}`

In [4]:
# Caso válido: Carcharodon carcharias -> match EXACT
tax_raw = fetch_gbif_taxonomy('Carcharodon carcharias')
print('>>> GBIF Species Match — Carcharodon carcharias')
print(f'HTTP {tax_raw["status"]}')
print(json.dumps(tax_raw['data'], indent=2))

>>> GBIF Species Match — Carcharodon carcharias
HTTP 200
{
  "usageKey": 2420694,
  "scientificName": "Carcharodon carcharias (Linnaeus, 1758)",
  "canonicalName": "Carcharodon carcharias",
  "rank": "SPECIES",
  "status": "ACCEPTED",
  "confidence": 99,
  "matchType": "EXACT",
  "kingdom": "Animalia",
  "phylum": "Chordata",
  "order": "Lamniformes",
  "family": "Lamnidae",
  "genus": "Carcharodon",
  "species": "Carcharodon carcharias",
  "kingdomKey": 1,
  "phylumKey": 44,
  "classKey": 121,
  "orderKey": 885,
  "familyKey": 5888,
  "genusKey": 2420693,
  "speciesKey": 2420694,
  "class": "Elasmobranchii"
}


In [5]:
# Caso inválido: Fakeus nonexistus -> NO_MATCH (será rechazado en Silver)
tax_bad = fetch_gbif_taxonomy('Fakeus nonexistus')
print('>>> GBIF Species Match — Fakeus nonexistus (sin match exacto)')
print(f'HTTP {tax_bad["status"]}')
if isinstance(tax_bad['data'], dict):
    print(json.dumps(tax_bad['data'], indent=2))
else:
    print(tax_bad['data'])

>>> GBIF Species Match — Fakeus nonexistus (sin match exacto)
HTTP 200
{
  "confidence": 100,
  "matchType": "NONE",
  "synonym": false
}


### 2.2 Query a GBIF Occurrence Search — ocurrencias

Endpoint: `GET https://api.gbif.org/v1/occurrence/search?scientificName={nombre}&hasCoordinate=true&limit=50`

In [6]:
# Caso válido: ocurrencias de Rhincodon typus (tiburón ballena)
occ_raw = fetch_gbif_occurrences('Rhincodon typus')
results = occ_raw['data'].get('results', [])
print('>>> GBIF Occurrence Search — Rhincodon typus')
print(f'HTTP {occ_raw["status"]}  |  Registros devueltos: {len(results)}\n')
for r in results[:3]:
    print(json.dumps({
        'gbifID': r.get('gbifID'),
        'country': r.get('country'),
        'decimalLatitude': r.get('decimalLatitude'),
        'decimalLongitude': r.get('decimalLongitude'),
        'eventDate': r.get('eventDate'),
        'basisOfRecord': r.get('basisOfRecord'),
        'iucnRedListCategory': r.get('iucnRedListCategory'),
    }, indent=2))
    print()

>>> GBIF Occurrence Search — Rhincodon typus
HTTP 200  |  Registros devueltos: 50

{
  "gbifID": "5938116839",
  "country": "Maldives",
  "decimalLatitude": 3.717267,
  "decimalLongitude": 72.758961,
  "eventDate": "2026-01-03T11:02",
  "basisOfRecord": "HUMAN_OBSERVATION",
  "iucnRedListCategory": "EN"
}

{
  "gbifID": "5938382709",
  "country": "Philippines",
  "decimalLatitude": 9.98467,
  "decimalLongitude": 125.279289,
  "eventDate": "2026-01-03T09:28:03",
  "basisOfRecord": "HUMAN_OBSERVATION",
  "iucnRedListCategory": "EN"
}

{
  "gbifID": "5938466615",
  "country": "Australia",
  "decimalLatitude": -16.070685,
  "decimalLongitude": 145.816655,
  "eventDate": "2026-01-03T11:33",
  "basisOfRecord": "HUMAN_OBSERVATION",
  "iucnRedListCategory": "EN"
}



In [7]:
# Otra especie: Carcharodon carcharias
occ_raw2 = fetch_gbif_occurrences('Carcharodon carcharias')
results2 = occ_raw2['data'].get('results', [])
print('>>> GBIF Occurrence Search — Carcharodon carcharias')
print(f'HTTP {occ_raw2["status"]}  |  Registros devueltos: {len(results2)}\n')
print(f'  Primer país:    {results2[0].get("country")}')
print(f'  Primeras coord: {results2[0].get("decimalLatitude")}, {results2[0].get("decimalLongitude")}')
print(f'  IUCN:           {results2[0].get("iucnRedListCategory")}')

>>> GBIF Occurrence Search — Carcharodon carcharias
HTTP 200  |  Registros devueltos: 50

  Primer país:    South Africa
  Primeras coord: -32.777707, 28.305201
  IUCN:           VU


### 2.3 Query a Wikipedia — summary + extracts

Dos endpoints por especie:
- `GET /api/rest_v1/page/summary/{title}` — párrafo introductorio
- `GET /w/api.php?action=query&prop=extracts&explaintext=true&titles={title}` — secciones completas

In [8]:
# Resolución de título: de nombre científico a título Wikipedia
print('>>> Resolución de títulos Wikipedia\n')
for nombre in ['Carcharodon carcharias', 'Great white shark', 'Fakeus nonexistus']:
    res = resolve_wikipedia_title(nombre)
    print(f'  Entrada: {nombre:30s} -> Título: {str(res.title):25s} | Status: {res.status.value}')

>>> Resolución de títulos Wikipedia

  Entrada: Carcharodon carcharias         -> Título: Great white shark         | Status: SUCCESS
  Entrada: Great white shark              -> Título: Great white shark         | Status: SUCCESS
  Entrada: Fakeus nonexistus              -> Título: None                      | Status: PAGE_NOT_FOUND


In [9]:
# Resumen Wikipedia
wp_summary = fetch_wikipedia_summary('Great white shark')
print('>>> Wikipedia Summary — Great white shark')
print(f'HTTP {wp_summary["status"]}')
print(f'Título:  {wp_summary["data"].get("title")}')
print(f'URL:     {wp_summary["data"].get("content_urls", {}).get("desktop", {}).get("page", "")}')
print(f'Extracto:\n{wp_summary["data"].get("extract", "")[:600]}...')

>>> Wikipedia Summary — Great white shark
HTTP 200
Título:  Great white shark
URL:     https://en.wikipedia.org/wiki/Great_white_shark
Extracto:
The great white shark, also known as the white shark, white pointer, or great white, is a large shark. It is closely related to the mako sharks, the porbeagle, and the salmon shark. It is a robustly built species with a grayish upperside and a white underside. The white shark is one of the largest living shark and fish species, though it is still smaller than the whale shark and basking shark. It has about 300 triangular, serrated teeth that are continuously replaced. Its massive, fatty liver can reach over a quarter of its body weight, providing buoyancy and storing energy. The species is par...


In [10]:
# Extractos completos para ver las secciones del artículo
wp_extracts = fetch_wikipedia_extracts('Great white shark')
pages = wp_extracts['data'].get('query', {}).get('pages', {})
for pid, pdata in pages.items():
    extract_text = pdata.get('extract', '')
    print(f'>>> Wikipedia Extracts — {pdata.get("title")}')
    print(f'Longitud total: {len(extract_text)} caracteres')
    print('Secciones detectadas:')
    for line in extract_text.split('\n'):
        if line.startswith('=='):
            print(f'  {line.strip()}')

>>> Wikipedia Extracts — Great white shark
Longitud total: 47182 caracteres
Secciones detectadas:
  == Etymology and naming ==
  == Taxonomy and evolution ==
  === Populations and genetic history ===
  == Appearance and anatomy ==
  === Size ===
  === Teeth and jaws ===
  === Senses ===
  === Internal physiology ===
  == Distribution and habitat ==
  === Migrations ===
  == Behavior and ecology ==
  === Diet and feeding ===
  === Social communication ===
  === Reproduction and growth ===
  === Mortality and health ===
  == Relationship with humans ==
  === Bites ===
  === Captivity ===
  === Tourism ===
  == Conservation ==
  === South Africa ===
  === Oceania ===
  === United States ===
  === Europe and the Mediterranean ===
  == See also ==
  == Notes ==
  == References ==
  === Sources ===
  == External links ==


### 2.4 Persistencia Bronze

Todo lo anterior se persiste como JSON crudo con envelope de metadatos. Nada se descarta.

In [11]:
# Persistir una consulta real a Bronze
tax_demo = fetch_gbif_taxonomy('Galeocerdo cuvier')
path = persist_bronze(
    source='gbif_taxonomy',
    lote=2,
    species_queried=['Galeocerdo cuvier'],
    raw_responses=[{'query': 'Galeocerdo cuvier', 'response': tax_demo}],
)
print(f'Archivo creado: {path}\n')
with open(path) as f:
    envelope = json.load(f)
print(json.dumps(envelope, indent=2)[:600] + '...')

Archivo creado: data/bronze/gbif_taxonomy/lote_2__2026-07-28T00-56-32Z.json

{
  "source": "gbif_taxonomy",
  "lote": 2,
  "ingested_at": "2026-07-28T00:56:32Z",
  "species_queried": [
    "Galeocerdo cuvier"
  ],
  "raw_responses": [
    {
      "query": "Galeocerdo cuvier",
      "response": {
        "status": 200,
        "data": {
          "usageKey": 2418234,
          "scientificName": "Galeocerdo cuvier (P\u00e9ron & Lesueur, 1822)",
          "canonicalName": "Galeocerdo cuvier",
          "rank": "SPECIES",
          "status": "ACCEPTED",
          "confidence": 99,
          "matchType": "EXACT",
          "kingdom": "Animalia",
          "phylum": "Chordat...


---
## 3. Silver Layer — Validación y contratos

### Propósito

Transformar el JSON crudo de Bronze en registros validados por **contratos Pydantic v2**. Cada fuente tiene su propio validador y su propia partición `valid/` / `rejected/`.

### Estructura de salida

```
data/silver/
├── wikipedia/valid/{species_key}.json
├── wikipedia/rejected/{batch_id}_wikipedia_{species_key}.json
├── gbif_taxonomy/valid/{species_key}.json
├── gbif_taxonomy/rejected/{batch_id}_gbif_taxonomy_{species_key}.json
├── gbif_occurrence/valid/{gbif_id}.json
└── gbif_occurrence/rejected/{batch_id}_gbif_occurrence_{gbif_id}.json
```

### Contratos (Pydantic v2)

In [12]:
# WikipediaPage: contrato pivote (1 por species_key)
print('WikipediaPage:')
for name, field in WikipediaPage.model_fields.items():
    print(f'  {name:20s} | {str(field.annotation):25s} | required={field.is_required()}')

WikipediaPage:
  species_key          | <class 'str'>             | required=True
  title                | <class 'str'>             | required=True
  resolved_via         | <class 'str'>             | required=True
  extract              | <class 'str'>             | required=True
  sections_text        | <class 'str'>             | required=True
  page_url             | <class 'str'>             | required=True
  is_disambiguation    | <class 'bool'>            | required=False


In [13]:
print('GBIFTaxonomy:')
for name, field in GBIFTaxonomy.model_fields.items():
    print(f'  {name:20s} | {str(field.annotation):25s} | required={field.is_required()}')

GBIFTaxonomy:
  species_key          | <class 'str'>             | required=True
  usage_key            | <class 'int'>             | required=True
  scientific_name      | <class 'str'>             | required=True
  canonical_name       | <class 'str'>             | required=True
  rank                 | <class 'str'>             | required=True
  match_type           | <class 'str'>             | required=True
  confidence           | <class 'int'>             | required=True
  kingdom              | typing.Optional[str]      | required=False
  phylum               | typing.Optional[str]      | required=False
  class_name           | typing.Optional[str]      | required=False


In [14]:
print('GBIFOccurrence:')
for name, field in GBIFOccurrence.model_fields.items():
    print(f'  {name:20s} | {str(field.annotation):25s} | required={field.is_required()}')

GBIFOccurrence:
  gbif_id              | <class 'int'>             | required=True
  species_key          | <class 'str'>             | required=True
  decimal_latitude     | <class 'float'>           | required=True
  decimal_longitude    | <class 'float'>           | required=True
  country              | typing.Optional[str]      | required=False
  event_date           | typing.Optional[datetime.datetime] | required=False
  basis_of_record      | <class 'str'>             | required=True
  iucn_red_list_category | typing.Optional[str]      | required=False
  coordinate_uncertainty_m | typing.Optional[float]    | required=False


### 3.1 Validación Wikipedia — válido vs rechazado

In [15]:
# CASO VÁLIDO: Great white shark
res = resolve_wikipedia_title('Carcharodon carcharias')
wp_summary = fetch_wikipedia_summary(res.title)
wp_extracts = fetch_wikipedia_extracts(res.title)
wp_valid = validate_wikipedia(
    {'raw': {'summary': wp_summary, 'extracts': wp_extracts}, 'resolved_via': res.resolved_via.value},
    batch_id='demo', species_key='carcharodon_carcharias',
)
print('>>> Wikipedia VÁLIDO' if wp_valid else '>>> Wikipedia RECHAZADO')
if wp_valid:
    print(f'  species_key:  {wp_valid["species_key"]}')
    print(f'  title:        {wp_valid["title"]}')
    print(f'  resolved_via: {wp_valid["resolved_via"]}')
    print(f'  extract_len:  {len(wp_valid["extract"])} chars')
    print(f'  sections_len: {len(wp_valid["sections_text"])} chars')

>>> Wikipedia VÁLIDO
  species_key:  carcharodon_carcharias
  title:        Great white shark
  resolved_via: REDIRECT
  extract_len:  684 chars
  sections_len: 47182 chars


In [16]:
# CASO RECHAZADO: especie inexistente -> PAGE_NOT_FOUND
res_bad = resolve_wikipedia_title('Fakeus nonexistus')
print(f'Status resolución: {res_bad.status.value}')
print('(No se puede validar Wikipedia — la especie se cuarentena directamente)')

Status resolución: PAGE_NOT_FOUND
(No se puede validar Wikipedia — la especie se cuarentena directamente)


### 3.2 Validación GBIF Taxonomy — válido vs rechazado

In [17]:
# CASO VÁLIDO: match EXACT
tax_valid = validate_gbif_taxonomy(
    {'raw': fetch_gbif_taxonomy('Carcharodon carcharias')},
    batch_id='demo', species_key='carcharodon_carcharias',
)
print('>>> GBIF Taxonomy VÁLIDO' if tax_valid else '>>> GBIF Taxonomy RECHAZADO')
if tax_valid:
    print(f'  usage_key:      {tax_valid["usage_key"]}')
    print(f'  canonical_name: {tax_valid["canonical_name"]}')
    print(f'  rank:           {tax_valid["rank"]}')
    print(f'  kingdom:        {tax_valid["kingdom"]}')
    print(f'  phylum:         {tax_valid["phylum"]}')
    print(f'  class_name:     {tax_valid["class_name"]}')
    print(f'  match_type:     {tax_valid["match_type"]}')

>>> GBIF Taxonomy VÁLIDO
  usage_key:      2420694
  canonical_name: Carcharodon carcharias
  rank:           SPECIES
  kingdom:        Animalia
  phylum:         Chordata
  class_name:     Elasmobranchii
  match_type:     EXACT


In [18]:
# CASO RECHAZADO: sin match exacto
tax_rejected = validate_gbif_taxonomy(
    {'raw': fetch_gbif_taxonomy('Fakeus nonexistus')},
    batch_id='demo', species_key='fakeus_nonexistus',
)
print('>>> GBIF Taxonomy VÁLIDO' if tax_rejected else '>>> GBIF Taxonomy RECHAZADO')
print('(rechazado por MATCH_TYPE_NOT_EXACT — la especie sigue siendo válida en Gold vía Wikipedia)')

>>> GBIF Taxonomy RECHAZADO
(rechazado por MATCH_TYPE_NOT_EXACT — la especie sigue siendo válida en Gold vía Wikipedia)


### 3.3 Validación GBIF Occurrence — válido vs rechazado

In [19]:
# CASO VÁLIDO: ocurrencia con todos los campos requeridos
occ_raw = fetch_gbif_occurrences('Carcharodon carcharias')
valid_occs = []
rejected_occs = []
for rec in occ_raw['data'].get('results', []):
    r = validate_gbif_occurrence(rec, batch_id='demo', species_key='carcharodon_carcharias')
    if r:
        valid_occs.append(r)
    else:
        rejected_occs.append(rec.get('gbifID', '?'))
print('>>> Validación de ocurrencias — Carcharodon carcharias')
print(f'  Válidas:   {len(valid_occs)}')
print(f'  Rechazadas:{len(rejected_occs)}')
print(f'  Total:     {len(occ_raw["data"].get("results", []))}')
print()
if valid_occs:
    print('Ejemplo de ocurrencia válida:')
    print(valid_occs[0])

>>> Validación de ocurrencias — Carcharodon carcharias
  Válidas:   43
  Rechazadas:7
  Total:     50

Ejemplo de ocurrencia válida:
{'gbif_id': 5938331810, 'species_key': 'carcharodon_carcharias', 'decimal_latitude': -32.777707, 'decimal_longitude': 28.305201, 'country': 'South Africa', 'event_date': datetime.datetime(2026, 1, 3, 5, 9, 36), 'basis_of_record': 'HUMAN_OBSERVATION', 'iucn_red_list_category': 'VU', 'coordinate_uncertainty_m': None}


In [20]:
# Mostrar todos los motivos de rechazo disponibles en Silver
print('Motivos de rechazo:')
for r in RejectionReason:
    print(f'  {r.value}')

Motivos de rechazo:
  PAGE_NOT_FOUND
  AMBIGUOUS_RESOLUTION
  EXTRACT_TOO_SHORT
  DISAMBIGUATION_PAGE
  HTTP_ERROR
  MISSING_SECTIONS_TEXT
  MATCH_TYPE_NOT_EXACT
  NO_MATCH_RETURNED
  MISSING_COORDINATES
  COORDINATES_OUT_OF_RANGE
  MISSING_GBIF_ID
  INVALID_EVENT_DATE
  MISSING_BASIS_OF_RECORD


---
## 4. Gold Layer — Conocimiento canónico y búsqueda

### Transición Silver → Gold

Entre Silver y Gold ocurren tres procesos intermedios:

1. **Merge** — Wikipedia se usa como pivote (1:1), GBIF Taxonomy se une si EXACT, Occurrence se agrega (N→1).
2. **Warehouse** — DuckDB embebido con 5 tablas, UPSERT idempotente por clave natural.
3. **Gold** — Se construyen las entidades Species, SpeciesChunk, SpeciesMetadata; se generan embeddings y el índice FAISS.

### Entidades Gold

- **Species (Parent)**: documento completo con 12 secciones canónicas.
- **SpeciesChunk (Child)**: fragmentos semánticos por sección, cada uno con embedding 384d.
- **SpeciesMetadata**: filtros estructurados (taxonomía, IUCN, países, años).

### 4.1 Merge — Integración Wikipedia + GBIF

Fusionamos los datos validados de las tres fuentes en un solo registro por especie.

In [21]:
# Pipeline manual de Silver -> Merge para una especie
sk = 'carcharodon_carcharias'

# 1. Silver: validar Wikipedia
res = resolve_wikipedia_title('Carcharodon carcharias')
wp_summary = fetch_wikipedia_summary(res.title)
wp_extracts = fetch_wikipedia_extracts(res.title)
wp_valid = validate_wikipedia(
    {'raw': {'summary': wp_summary, 'extracts': wp_extracts},
     'resolved_via': res.resolved_via.value},
    batch_id='demo', species_key=sk,
)

# 2. Silver: validar GBIF Taxonomy
tax_valid = validate_gbif_taxonomy(
    {'raw': fetch_gbif_taxonomy('Carcharodon carcharias')},
    batch_id='demo', species_key=sk,
)

# 3. Silver: validar ocurrencias
occ_valid = []
for rec in fetch_gbif_occurrences('Carcharodon carcharias')['data'].get('results', []):
    r = validate_gbif_occurrence(rec, batch_id='demo', species_key=sk)
    if r:
        occ_valid.append(r)

# 4. Merge: fusionar fuentes
merged = merge_species(wp_valid, tax_valid, occ_valid)
if merged:
    print('>>> Merge completo para Carcharodon carcharias')
    print(json.dumps(merged.model_dump(), indent=2, default=str))

>>> Merge completo para Carcharodon carcharias
{
  "species_key": "carcharodon_carcharias",
  "wikipedia_title": "Great white shark",
  "extract": "The great white shark, also known as the white shark, white pointer, or great white, is a large shark. It is closely related to the mako sharks, the porbeagle, and the salmon shark. It is a robustly built species with a grayish upperside and a white underside. The white shark is one of the largest living shark and fish species, though it is still smaller than the whale shark and basking shark. It has about 300 triangular, serrated teeth that are continuously replaced. Its massive, fatty liver can reach over a quarter of its body weight, providing buoyancy and storing energy. The species is partially warm-blooded, an adaptation that allows it to remain active in colder waters.",
  "sections_text": "The great white shark (Carcharodon carcharias), also known as the white shark, white pointer, or great white, is a large shark. It is closely rel

### 4.2 Warehouse — Carga en DuckDB

Los registros fusionados se cargan en DuckDB con UPSERT idempotente.

In [22]:
# Inicializar Warehouse y cargar datos
with Warehouse() as wh:
    wh.initialize_schema()

    # Cargar Wikipedia validada
    wh.upsert_wikipedia([wp_valid])
    # Cargar taxonomía validada
    wh.upsert_gbif_taxonomy([tax_valid])
    # Cargar ocurrencias validadas
    wh.upsert_gbif_occurrences(occ_valid)
    # Cargar merged
    wh.upsert_species_merged([merged.model_dump()])
    # Registrar en catálogo
    wh.upsert_species_catalog([{
        'species_key': sk,
        'scientific_name': 'Carcharodon carcharias',
        'wikipedia_title': merged.wikipedia_title,
        'origin': 'SEED',
    }], batch_id='demo_medallones')

    print('>>> Evidence del Warehouse tras la carga:')
    for e in wh.get_evidence():
        print(f'  {e["tabla"]:30s} | {e["count"]} filas')

>>> Evidence del Warehouse tras la carga:
  silver_wikipedia               | 2 filas
  silver_gbif_taxonomy           | 2 filas
  silver_gbif_occurrences        | 68 filas
  species_merged                 | 2 filas
  species_catalog                | 2 filas


In [23]:
# Consultar species_merged directamente desde DuckDB
with Warehouse() as wh:
    conn = wh.connect()
    row = conn.execute(
        'SELECT species_key, wikipedia_title, taxonomy_status, n_occurrences_validas, iucn_red_list_category, augmentation_status, paises '
        'FROM species_merged WHERE species_key = ?', [sk]
    ).fetchone()
    columns = ['species_key', 'wikipedia_title', 'taxonomy_status', 'n_occurrences_validas', 'iucn_red_list_category', 'augmentation_status', 'paises']
    if row:
        print('>>> Consulta directa a species_merged:')
        for c, v in zip(columns, row):
            print(f'  {c:30s} | {v}')

>>> Consulta directa a species_merged:
  species_key                    | carcharodon_carcharias
  wikipedia_title                | Great white shark
  taxonomy_status                | MATCHED
  n_occurrences_validas          | 43
  iucn_red_list_category         | VU
  augmentation_status            | OK
  paises                         | ['United States of America', 'South Africa', 'Australia']


### 4.3 Gold — Chunking semántico

A partir del merged construimos el modelo **Species** con 12 secciones canónicas y lo fragmentamos en **SpeciesChunks**.

Secciones canónicas:
Identity, Taxonomy, Morphology, Distribution, Habitat, Behavior, Diet, Reproduction, Conservation, Human Interaction, References, Multimedia.

In [24]:
print('Species — 12 secciones canónicas:')
for name, field in Species.model_fields.items():
    print(f'  {name:20s} | {field.annotation}')

Species — 12 secciones canónicas:
  species_key          | <class 'str'>
  identity             | typing.Optional[str]
  taxonomy             | typing.Optional[str]
  morphology           | typing.Optional[str]
  distribution         | typing.Optional[str]
  habitat              | typing.Optional[str]
  behavior             | typing.Optional[str]
  diet                 | typing.Optional[str]
  reproduction         | typing.Optional[str]
  conservation         | typing.Optional[str]
  human_interaction    | typing.Optional[str]
  references           | typing.Optional[str]
  multimedia           | typing.Optional[str]


In [25]:
# Construir Species desde merged y generar chunks
species_dict = build_species_from_merged(merged.model_dump())
chunks = chunk_species(species_dict)

print('>>> Species construido:')
for k, v in species_dict.items():
    if k != 'species_key':
        status = '✅' if v else '❌'
        print(f'  {status} {k:20s} | {"presente" if v else "vacío"}')

print(f'\n>>> Chunks generados: {len(chunks)}')
for ch in chunks:
    print(f'  {ch["chunk_id"]}  ({len(ch["text"])} chars)')

>>> Species construido:
  ✅ identity             | presente
  ✅ taxonomy             | presente
  ✅ morphology           | presente
  ✅ distribution         | presente
  ❌ habitat              | vacío
  ✅ behavior             | presente
  ✅ diet                 | presente
  ✅ reproduction         | presente
  ✅ conservation         | presente
  ✅ human_interaction    | presente
  ✅ references           | presente
  ❌ multimedia           | vacío

>>> Chunks generados: 10
  carcharodon_carcharias__Identity__0  (51 chars)
  carcharodon_carcharias__Taxonomy__0  (78 chars)
  carcharodon_carcharias__Morphology__0  (891 chars)
  carcharodon_carcharias__Distribution__0  (1460 chars)
  carcharodon_carcharias__Behavior__0  (1514 chars)
  carcharodon_carcharias__Diet__0  (6360 chars)
  carcharodon_carcharias__Reproduction__0  (2936 chars)
  carcharodon_carcharias__Conservation__0  (2936 chars)
  carcharodon_carcharias__Human Interaction__0  (1098 chars)
  carcharodon_carcharias__References__0  (

### 4.4 Búsqueda híbrida con filtros

La función `search()` implementa retrieval Parent-Child:
1. Embed query con `BAAI/bge-small-en-v1.5`
2. FAISS oversampling (`top_k * 5`)
3. Dedup por species_key (mejor score)
4. Filtros estructurados opcionales: taxonomía, IUCN, país

**Nota**: La búsqueda requiere un índice FAISS preconstruido (correr el pipeline completo). Estos ejemplos muestran la firma y el uso esperado con filtros híbridos.

In [26]:
# Demostración de la firma de search() con filtros
print('Firma de search():')
import inspect
sig = inspect.signature(search)
for name, param in sig.parameters.items():
    default = f' = {param.default}' if param.default is not inspect.Parameter.empty else ''
    print(f'  {name}: {param.annotation}{default}')

Firma de search():
  query: str
  top_k: int = 5
  filtro_taxonomia: Optional[dict] = None
  filtro_iucn: Optional[str] = None
  filtro_pais: Optional[str] = None


In [27]:
# Ejemplo de llamada a search (requiere índice FAISS pre-construido)
print('Ejemplos de consulta híbrida con filtros:\n')
print('>>> Sin filtros:')
print('  search(query="large coastal shark", top_k=3)')
print('>>> Con filtro taxonómico:')
print('  search(')
print('      query="shark that eats plankton",')
print('      top_k=3,')
print('      filtro_taxonomia={"class_name": "Chondrichthyes"},')
print('  )')
print('>>> Con filtro IUCN:')
print('  search(')
print('      query="endangered shark species",')
print('      top_k=5,')
print('      filtro_iucn="ENDANGERED",')
print('  )')
print('>>> Con filtro geográfico:')
print('  search(')
print('      query="sharks found near Australia",')
print('      top_k=3,')
print('      filtro_pais="Australia",')
print('  )')
print('>>> Múltiples filtros combinados (AND):')
print('  search(')
print('      query="reef shark habitat",')
print('      top_k=3,')
print('      filtro_taxonomia={"class_name": "Chondrichthyes"},')
print('      filtro_pais="Mexico",')
print('  )')

Ejemplos de consulta híbrida con filtros:

>>> Sin filtros:
  search(query="large coastal shark", top_k=3)
>>> Con filtro taxonómico:
  search(
      query="shark that eats plankton",
      top_k=3,
      filtro_taxonomia={"class_name": "Chondrichthyes"},
  )
>>> Con filtro IUCN:
  search(
      query="endangered shark species",
      top_k=5,
      filtro_iucn="ENDANGERED",
  )
>>> Con filtro geográfico:
  search(
      query="sharks found near Australia",
      top_k=3,
      filtro_pais="Australia",
  )
>>> Múltiples filtros combinados (AND):
  search(
      query="reef shark habitat",
      top_k=3,
      filtro_taxonomia={"class_name": "Chondrichthyes"},
      filtro_pais="Mexico",
  )


### 4.5 SpeciesMetadata — Filtros estructurados

Cada especie tiene metadatos asociados que permiten filtrar resultados por taxonomía, estado de conservación o geografía.

In [28]:
print('SpeciesMetadata:')
for name, field in SpeciesMetadata.model_fields.items():
    print(f'  {name:25s} | {field.annotation}')

SpeciesMetadata:
  species_key               | <class 'str'>
  kingdom                   | typing.Optional[str]
  phylum                    | typing.Optional[str]
  class_name                | typing.Optional[str]
  rank                      | typing.Optional[str]
  taxonomy_status           | <class 'str'>
  iucn_red_list_category    | typing.Optional[str]
  paises                    | list[str]
  anio_min                  | typing.Optional[int]
  anio_max                  | typing.Optional[int]
  n_occurrences_validas     | <class 'int'>
  augmentation_status       | <class 'str'>


---
## Resumen de la transición de datos

| Capa | Formato | Qué ocurre |
|------|---------|------------|
| **Bronze** | JSON crudo + envelope | Se persiste la respuesta exacta de cada API, sin transformación |
| **Silver** | JSON validado (Pydantic) | Se validan contratos; válidos van a `valid/`, inválidos a `rejected/` con motivo explícito |
| **Merge** | Registro fusionado | Wikipedia pivote 1:1 + GBIF Taxonomy (si EXACT) + Occurrence agregada (N→1) |
| **Warehouse** | DuckDB (5 tablas) | UPSERT idempotente por clave natural; `species_catalog` como registro maestro |
| **Gold** | FAISS + Parquet | Species (12 secciones), SpeciesChunks (embeddings 384d), SpeciesMetadata (filtros) + `search()` híbrido |

---
*Shark Knowledge Medallion — Composición y transición de datos a través de la arquitectura medallón*